# Part 1: Practise

## Task 1 : The mean and the median

In [1]:
import pandas as pd

date_columns = ['last_scraped', 'calendar_last_scraped', 'first_review', 'last_review']
listings = pd.read_csv("data/listings_clean.csv", parse_dates=date_columns)
print(listings.shape)

(4574, 73)


### Five small numbers of reviews, close to each other

In [3]:
# pd.Series([...]) makes one small column from your own numbers.
small = pd.Series([3, 5, 8, 10, 12])
print("mean:", small.mean())
print("median:", small.median())

mean: 7.6
median: 8.0


### Add the most popular listing (1293 reviews) and calculate again

In [4]:
with_big = pd.Series([3, 5, 8, 10, 12, 1293])
print("mean:", round(with_big.mean(), 1))
print("median:", with_big.median())

mean: 221.8
median: 9.0


### Mean and median of the real column, all 4574 listings

In [5]:
print("mean:", round(listings["number_of_reviews"].mean(), 1))
print("median:", listings["number_of_reviews"].median())

mean: 35.8
median: 9.0


### Answer
- Five numbers: mean 7.6, median 8.0.
- With 1293 added: the mean jumps to 221.8, but the median only moves to 9.0.
  One huge number pulls the mean up a lot. The median only looks at the middle, so it hardly moves.
- Real column: mean 35.8, median 9.0.
- For "how many reviews does a typical listing have?" I use the **median (9)**.
  Half of the listings have 9 reviews or fewer. The mean (35.8) is pulled up by a few very popular listings, so it is too high for a typical listing.

## Task 2 : Percentiles and the spread

### describe() gives all the main numbers at once

In [6]:
print(listings["number_of_reviews"].describe().round(1))

count    4574.0
mean       35.8
std        78.4
min         0.0
25%         2.0
50%         9.0
75%        29.0
max      1293.0
Name: number_of_reviews, dtype: float64


### Percentiles: the 10th, 25th, 50th, 75th, 90th and 99th

In [7]:
print(listings["number_of_reviews"].quantile([0.1, 0.25, 0.5, 0.75, 0.9, 0.99]))

0.10      0.00
0.25      2.00
0.50      9.00
0.75     29.00
0.90     94.00
0.99    400.27
Name: number_of_reviews, dtype: float64


### q1, q3 and the IQR (the middle half of the listings)

In [8]:
q1 = listings["number_of_reviews"].quantile(0.25)
q3 = listings["number_of_reviews"].quantile(0.75)
iqr = q3 - q1
print("q1:", q1, " q3:", q3, " iqr:", iqr)

q1: 2.0  q3: 29.0  iqr: 27.0


### Answer
- describe(): count 4574, mean 35.8, std 78.4, min 0, 25% 2, median 9, 75% 29, max 1293.
- Percentiles: 10% = 0, 25% = 2, 50% = 9, 75% = 29, 90% = 94, 99% = 400.
- q1 = 2, q3 = 29, IQR = 27.
- **IQR** the middle half of the listings have between 2 and 29 reviews, a spread of 27 reviews.
- **10th percentile = 0:** at least 10 percent of the listings (about 457) have no reviews at all.
- The jump from 90% (94) to 99% (400) shows the long tail: a few listings have very many reviews.

## Task 3 : The fences and the outliers

### Upper and lower fence with the IQR rule

In [9]:
upper = q3 + 1.5 * iqr
lower = q1 - 1.5 * iqr
print("upper:", upper, " lower:", lower)

upper: 69.5  lower: -38.5


### How many listings are above the upper fence

In [10]:
above = listings[listings["number_of_reviews"] > upper]
print(len(above))
print(round(len(above) / len(listings) * 100, 1), "percent")

582
12.7 percent


### Room types of the outliers

In [11]:
print(above["room_type"].value_counts())

room_type
Entire home/apt    446
Private room       121
Shared room         14
Hotel room           1
Name: count, dtype: int64


### Median price: outliers vs all the other listings

In [12]:
others = listings[listings["number_of_reviews"] <= upper]
print("outliers:", above["price"].median())
print("others:", others["price"].median())

outliers: 1459.54
others: 1867.0


### Answer
- Upper fence 69.5, lower fence -38.5.
- No outliers below: a listing cannot have a negative number of reviews, so nothing is under -38.5.
- 582 listings are above the upper fence (12.7 percent).
- Room types of the outliers: Entire home/apt 446, Private room 121, Shared room 14, Hotel room 1.
- Median price: outliers 1459.5 kr, all others 1867.0 kr.
  So the listings with the most reviews are **cheaper** than the rest. Cheaper places get booked more often, so they collect more reviews.

## Task 4 : Mark them, and compare

### Mark the outliers in a new column (do not remove them)

In [13]:
listings["reviews_outlier"] = listings["number_of_reviews"] > upper
print(listings["reviews_outlier"].sum())

582


### Table: all listings vs without the outliers

In [14]:
all_rev = listings["number_of_reviews"]
without = listings[~listings["reviews_outlier"]]["number_of_reviews"]

compare = pd.DataFrame({
    "all": all_rev.agg(["count", "mean", "median", "std"]),
    "without": without.agg(["count", "mean", "median", "std"])
})
compare["change %"] = (compare["without"] - compare["all"]) / compare["all"] * 100
print(compare.round(1))

           all  without  change %
count   4574.0   3992.0     -12.7
mean      35.8     12.9     -63.8
median     9.0      7.0     -22.2
std       78.4     15.5     -80.2


### Answer
- reviews_outlier is True for 582 listings.
- All listings: count 4574, mean 35.8, median 9.0, std 78.4.
- Without the outliers: count 3992, mean 12.9, median 7.0, std 15.5.
- Changes: count -13 %, mean -64 %, median -22 %, std -80 %.
- The **std changes the most**: the outliers made the spread look huge.
- The **median changes the least**: it only looks at the middle, so the outliers hardly affect it.

## Task 5 : Describe the groups

### All the numbers of number_of_reviews for every room type

In [15]:
print(listings.groupby("room_type")["number_of_reviews"].describe().round(0))

                  count   mean    std   min   25%   50%    75%     max
room_type                                                             
Entire home/apt  3760.0   34.0   76.0   0.0   2.0   8.0   27.0  1293.0
Hotel room          3.0   61.0   16.0  49.0  52.0  56.0   68.0    79.0
Private room      787.0   41.0   84.0   0.0   2.0  12.0   38.0   867.0
Shared room        24.0  114.0  122.0   0.0  28.0  82.0  146.0   510.0


### Answer
- Entire home/apt: count 3760, median 8.
- Hotel room: count 3, median 56.
- Private room: count 787, median 12.
- Shared room: count 24, median 82 (mean 114, max 510).
- **Shared room has the highest median (82)** among the groups with enough listings.
- I do **not** want to say anything about **Hotel room**: there are only 3 listings.
  3 listings is too few. One new hotel could change the median completely.

## Task 6 : Correlation

### number_of_reviews against 7 other columns

In [16]:
cols = ["number_of_reviews", "price", "accommodates", "bedrooms", "availability_365",
        "review_scores_rating", "minimum_nights", "days_since_review"]
print(listings[cols].corr()["number_of_reviews"].round(2))

number_of_reviews       1.00
price                  -0.09
accommodates           -0.05
bedrooms               -0.14
availability_365        0.18
review_scores_rating   -0.04
minimum_nights         -0.07
days_since_review      -0.19
Name: number_of_reviews, dtype: float64


### availability_365: with and without the outliers

In [17]:
normal = listings[~listings["reviews_outlier"]]

print("with outliers:", listings["number_of_reviews"].corr(listings["availability_365"]).round(2))
print("without outliers:", normal["number_of_reviews"].corr(normal["availability_365"]).round(2))

with outliers: 0.18
without outliers: 0.0


### Answer
- Correlation with number_of_reviews: price -0.09, accommodates -0.05, bedrooms -0.14,
  availability_365 0.18, rating -0.04, minimum_nights -0.07, days_since_review -0.19.
- All are **weak** (close to 0).
- Strongest: **days_since_review (-0.19)**. More reviews means the last review was more recent.
- Weakest: **rating (-0.04)**.
- Negative: all except availability_365.
- availability_365: **0.18 with outliers, 0.00 without**.
  The connection came only from the outliers, so 0.18 is misleading.